# Drawing: A New Way to Search: full QuickDraw training notebook
Train, test and predict on Google's **QuickDraw** dataset: **all 345 categories**, 28×28 bitmaps. Everything (training data, test data, predictions) comes from the dataset itself, no outside images.

**Run:** `Runtime > Change runtime type > T4 GPU`, then `Runtime > Run all`.

**About "all the data":** the complete dataset is ~50 million drawings (~40 GB as bitmaps), far more than a Colab session can train on in reasonable time. The notebook therefore uses **every category** and `PER_CLASS` drawings from each one (default 10,000, i.e. 3.45 million images; downloads only that slice, ~2.7 GB). Set `PER_CLASS = None` to use *every* drawing; that needs ~40 GB of disk and many hours per epoch.

## 1. Configuration

In [ ]:
PER_CLASS = 10_000   # drawings per category. 1_000 = quick dry run | None = every drawing (~50M, ~40 GB)
ONLY_CLASSES = None  # None = all 345 categories. Or an int (e.g. 50) for a random subset
EPOCHS = 12
BATCH = 512
SEED = 229
CACHE = "/content/qd_cache"   # downloaded subset is stored here (re-runs resume instead of re-downloading)

In [ ]:
import os, io, json, threading, urllib.parse, urllib.request
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from concurrent.futures import ThreadPoolExecutor
from tqdm.auto import tqdm

keras.utils.set_random_seed(SEED)
GPUS = tf.config.list_physical_devices("GPU")
print("TensorFlow", tf.__version__, "| GPUs:", GPUS)
if not GPUS:
    print("WARNING: no GPU. Runtime > Change runtime type > T4 GPU (training will be very slow on CPU).")
else:
    keras.mixed_precision.set_global_policy("mixed_float16")   # ~2x faster on T4

## 2. Get the category list

In [ ]:
CATS_URL = "https://raw.githubusercontent.com/googlecreativelab/quickdraw-dataset/master/categories.txt"
CLASSES = [l.strip() for l in urllib.request.urlopen(CATS_URL).read().decode().splitlines() if l.strip()]
if ONLY_CLASSES:
    pick = np.random.default_rng(SEED).permutation(len(CLASSES))[:ONLY_CLASSES]
    CLASSES = [CLASSES[i] for i in sorted(pick)]
K = len(CLASSES)
print(K, "categories, e.g.", CLASSES[:8])

## 3. Download the dataset
Each category is one `.npy` file on Google Cloud Storage. Only the first `PER_CLASS` rows of each file are streamed (not the whole file), 16 categories in parallel, shuffled within the category and written to one memory-mapped array on disk. If the runtime disconnects, run the cell again: finished categories are skipped.

In [ ]:
BASE_URL = "https://storage.googleapis.com/quickdraw_dataset/full/numpy_bitmap"
os.makedirs(CACHE, exist_ok=True)
X_PATH, META_PATH = f"{CACHE}/X.npy", f"{CACHE}/meta.json"
PIX = 784

def url_for(name): return f"{BASE_URL}/{urllib.parse.quote(name)}.npy"

def _open(url, nbytes=None):
    headers = {"Range": f"bytes=0-{nbytes - 1}"} if nbytes else {}
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=120)

def _retry(fn, tries=4):
    for t in range(tries):
        try: return fn()
        except Exception as e:
            if t == tries - 1: raise
            import time; time.sleep(2 * (t + 1))

def npy_info(name):
    """(number of rows, header size in bytes) read from the first bytes of the .npy file."""
    def go():
        with _open(url_for(name), 512) as r:
            buf = io.BytesIO(r.read(512))
        ver = np.lib.format.read_magic(buf)
        shape, _, _ = (np.lib.format.read_array_header_1_0(buf) if ver == (1, 0)
                       else np.lib.format.read_array_header_2_0(buf))
        return shape[0], buf.tell()
    return _retry(go)

def fetch_rows(name, n_rows, header):
    def go():
        need = n_rows * PIX
        out = bytearray(need); mv = memoryview(out); got = 0
        with _open(url_for(name), header + need if PER_CLASS else None) as r:
            r.read(header)
            while got < need:
                k = r.readinto(mv[got:])
                if not k: raise IOError("connection closed early")
                got += k
        return np.frombuffer(out, np.uint8).reshape(n_rows, PIX)
    return _retry(go)

# phase 1: row counts (tiny requests)
with ThreadPoolExecutor(16) as ex:
    infos = list(tqdm(ex.map(npy_info, CLASSES), total=K, desc="reading headers"))
avail = [n for n, _ in infos]; headers = [h for _, h in infos]
counts = [min(n, PER_CLASS) if PER_CLASS else n for n in avail]
starts = np.concatenate([[0], np.cumsum(counts)]).astype(np.int64)
TOTAL = int(starts[-1])
print(f"{K} categories, {TOTAL:,} drawings ({TOTAL * PIX / 1e9:.1f} GB on disk)")

# phase 2: download into one memmap (resumable)
sig = {"classes": CLASSES, "counts": counts}
done = set()
if os.path.exists(META_PATH) and os.path.exists(X_PATH):
    meta = json.load(open(META_PATH))
    if meta["sig"] == sig: done = set(meta["done"])
X = (np.load(X_PATH, mmap_mode="r+") if done else
     np.lib.format.open_memmap(X_PATH, mode="w+", dtype=np.uint8, shape=(TOTAL, PIX)))
lock = threading.Lock()

def work(i):
    if i in done: return
    data = fetch_rows(CLASSES[i], counts[i], headers[i])
    data = data[np.random.default_rng(SEED + i).permutation(len(data))]   # shuffle within category
    X[starts[i]:starts[i + 1]] = data
    with lock:
        done.add(i)
        json.dump({"sig": sig, "done": sorted(done)}, open(META_PATH, "w"))

with ThreadPoolExecutor(16) as ex:
    list(tqdm(ex.map(work, range(K)), total=K, desc="downloading categories"))
X.flush()
y_all = np.repeat(np.arange(K, dtype=np.int16), counts)
print("ready:", X.shape, "labels:", y_all.shape)

## 4. Look at the data

In [ ]:
fig, axes = plt.subplots(6, 10, figsize=(13, 8.5))
for ax in axes.ravel(): ax.axis("off")
for ax, i in zip(axes.ravel(), np.random.default_rng(1).choice(K, min(60, K), replace=False)):
    j = starts[i] + np.random.default_rng(i).integers(counts[i])
    ax.imshow(np.asarray(X[j]).reshape(28, 28), cmap="gray_r"); ax.set_title(CLASSES[i], fontsize=8); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Train / validation / test split
80 / 10 / 10 **within every category** (stratified), so all 345 categories appear in each split.

In [ ]:
tr, va, te = [], [], []
for s, e in zip(starts[:-1], starts[1:]):
    n = e - s; a, b = int(0.8 * n), int(0.9 * n)
    tr.append(np.arange(s, s + a)); va.append(np.arange(s + a, s + b)); te.append(np.arange(s + b, e))
tr_idx, va_idx, te_idx = map(np.concatenate, (tr, va, te))
print(f"train {len(tr_idx):,} | validation {len(va_idx):,} | test {len(te_idx):,}")

_Base = getattr(keras.utils, "PyDataset", None) or keras.utils.Sequence
class Batches(_Base):
    """Feeds uint8 batches straight from the on-disk array (scaling to [0,1] happens inside the model)."""
    def __init__(self, idx, shuffle):
        super().__init__(**({"workers": 2, "max_queue_size": 16} if _Base.__name__ == "PyDataset" else {}))
        self.idx, self.shuffle, self.rng = idx.copy(), shuffle, np.random.default_rng(SEED)
        if shuffle: self.rng.shuffle(self.idx)
    def __len__(self): return int(np.ceil(len(self.idx) / BATCH))
    def __getitem__(self, i):
        ix = np.sort(self.idx[i * BATCH:(i + 1) * BATCH])
        return X[ix].reshape(-1, 28, 28, 1), y_all[ix].astype(np.int32)
    def on_epoch_end(self):
        if self.shuffle: self.rng.shuffle(self.idx)

train_seq, val_seq, test_seq = Batches(tr_idx, True), Batches(va_idx, False), Batches(te_idx, False)

## 6. Model
A convolutional network in the spirit of the report's findings (plain conv + max-pool blocks, no exotic layers), widened for 345 categories.

In [ ]:
def build_model(n_classes):
    L = keras.layers
    m = keras.Sequential([
        L.Input(shape=(28, 28, 1), dtype="uint8"),
        L.Rescaling(1 / 255.0),
        L.Conv2D(64, 3, padding="same", activation="relu"), L.BatchNormalization(), L.MaxPooling2D(),
        L.Conv2D(128, 3, padding="same", activation="relu"), L.BatchNormalization(), L.MaxPooling2D(),
        L.Conv2D(256, 3, padding="same", activation="relu"), L.BatchNormalization(), L.MaxPooling2D(),
        L.Flatten(), L.Dense(512, activation="relu"), L.Dropout(0.3),
        L.Dense(n_classes, activation="softmax", dtype="float32"),
    ], name="doodle_cnn")
    m.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy",
              metrics=["accuracy", keras.metrics.SparseTopKCategoricalAccuracy(5, name="top5")])
    return m
model = build_model(K); model.summary()

## 7. Train

In [ ]:
import time
callbacks = [
    keras.callbacks.ModelCheckpoint("/content/best.keras", monitor="val_accuracy", save_best_only=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_accuracy", factor=0.5, patience=1, min_lr=1e-5, verbose=1),
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
]
t0 = time.time()
history = model.fit(train_seq, validation_data=val_seq, epochs=EPOCHS, callbacks=callbacks, verbose=2)
print(f"training time: {(time.time() - t0) / 60:.1f} min")

In [ ]:
h = history.history
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(h["loss"], label="train"); ax[0].plot(h["val_loss"], label="validation"); ax[0].set_title("Loss"); ax[0].legend()
ax[1].plot(h["accuracy"], label="train"); ax[1].plot(h["val_accuracy"], label="validation"); ax[1].set_title("Top-1 accuracy"); ax[1].legend()
for a in ax: a.set_xlabel("Epoch")
plt.tight_layout(); plt.show()

## 8. Test on held-out drawings

In [ ]:
test_loss, test_acc, test_top5 = model.evaluate(test_seq, verbose=0)
print(f"TEST  top-1 accuracy: {test_acc:.2%} | top-5 accuracy: {test_top5:.2%}   (chance = {1 / K:.2%})")

probs = model.predict(test_seq, verbose=0)               # same order as te_idx
y_true, y_pred = y_all[te_idx].astype(int), probs.argmax(1)
cm = np.zeros((K, K), dtype=np.int32); np.add.at(cm, (y_true, y_pred), 1)
per_class = cm.diagonal() / cm.sum(1).clip(min=1)
order = np.argsort(per_class)
print("\nHardest categories: ", ", ".join(f"{CLASSES[i]} ({per_class[i]:.0%})" for i in order[:10]))
print("Easiest categories: ", ", ".join(f"{CLASSES[i]} ({per_class[i]:.0%})" for i in order[::-1][:10]))

off = cm.copy(); np.fill_diagonal(off, 0)
top = np.dstack(np.unravel_index(np.argsort(off.ravel())[::-1][:15], off.shape))[0]
pd.DataFrame([(CLASSES[i], CLASSES[j], off[i, j] / cm[i].sum()) for i, j in top],
             columns=["true", "predicted as", "confusion rate"]).style.format({"confusion rate": "{:.1%}"})

## 9. Predict on drawings from the dataset
Random drawings from the **test split** (never seen in training): the model's top-3 guesses, green = correct, red = wrong.

In [ ]:
def show_predictions(sample_idx):
    """sample_idx: positions in the test split (0..len(te_idx)-1)."""
    n = len(sample_idx); cols = min(6, n); rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(2.4 * cols, 2.9 * rows), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for ax, p in zip(axes.ravel(), sample_idx):
        t, pr = y_true[p], probs[p]; top3 = np.argsort(pr)[::-1][:3]
        ax.imshow(np.asarray(X[te_idx[p]]).reshape(28, 28), cmap="gray_r")
        ax.set_title(f"true: {CLASSES[t]}", fontsize=9)
        ax.text(0.5, -0.08, "\n".join(f"{CLASSES[c]} {pr[c]:.0%}" for c in top3), transform=ax.transAxes,
                ha="center", va="top", fontsize=8, color="green" if top3[0] == t else "crimson")
    plt.tight_layout(); plt.show()

show_predictions(np.random.default_rng().choice(len(te_idx), 12, replace=False))

In [ ]:
def predict_class(name="banana", n=6):
    """Predict n random held-out test drawings of one category."""
    c = CLASSES.index(name)
    pos = np.flatnonzero(y_true == c)
    show_predictions(np.random.default_rng().choice(pos, min(n, len(pos)), replace=False))

predict_class("banana")          # change the category name to try others

In [ ]:
# Optional: pick a category from a dropdown
import ipywidgets as widgets
widgets.interact(predict_class, name=widgets.Dropdown(options=sorted(CLASSES), value=sorted(CLASSES)[0]),
                 n=widgets.IntSlider(value=6, min=1, max=12))

## 10. Save the model

In [ ]:
model.save("/content/doodle_cnn_345.keras")
json.dump(CLASSES, open("/content/doodle_classes.json", "w"))
from google.colab import files
files.download("/content/doodle_cnn_345.keras"); files.download("/content/doodle_classes.json")